# Measure It · optional T4 reproduction
The supplied CPU checkpoint can run inference immediately. Use this notebook only to reproduce training on your own Colab GPU. Select a GPU runtime and run the cells once. If the image folder is missing, upload the prepared Envelope-Dataset.zip when prompted. The fixed split files and reviewed masks come from the same run as the local project.


In [ ]:
from pathlib import Path
import subprocess, sys, os, shutil
from google.colab import files

root = Path("/content/measure-it-envelopes")
if not root.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "dev-test01",
                    "https://github.com/Sherry-27/measure-it-envelopes.git", str(root)], check=True)
os.chdir(root)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Project revision", commit)
import hashlib, zipfile
from pathlib import PurePosixPath
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Select a GPU runtime before running this notebook.")
requirements = []
for split in ("train", "val", "test"):
    for name in (root/"dataset"/"splits"/f"{split}.txt").read_text().splitlines():
        requirements += [f"dataset/images/{name}", f"dataset/masks/{Path(name).with_suffix('.png')}"]
if not all((root/entry).is_file() for entry in requirements):
    print("Upload the prepared Envelope-Dataset.zip (70 images and 70 masks).")
    upload = files.upload()
    if len(upload) != 1:
        raise ValueError("Upload one ZIP file.")
    import io
    with zipfile.ZipFile(io.BytesIO(next(iter(upload.values())))) as archive:
        for entry in requirements:
            target = root/entry
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(entry))
        for split in ("train", "val", "test"):
            index = f"dataset/splits/{split}.txt"
            if archive.read(index).decode().splitlines() != (root/index).read_text().splitlines():
                raise ValueError(f"Split mismatch for {split}")
print("Ready:", len(requirements)//2, "image/mask pairs.")


In [ ]:
env = Path("/content/measure-it-venv")
if not env.exists():
    subprocess.run([sys.executable, "-m", "venv", str(env)], check=True)
python = str(env/"bin"/"python")
def run(*args):
    return subprocess.run([python, *args], cwd=root, check=True)
run("-m", "pip", "install", "-r", "requirements.txt")
run("-c", "import torch; assert torch.cuda.is_available(), 'Select a GPU runtime'; print(torch.cuda.get_device_name(0))")

In [ ]:
run("-m", "models.train", "--device", "cuda")
bundle = root/"models"/"runs"/"first"
(bundle/"git-revision.txt").write_text(commit+"\n")
(bundle/"pip-freeze.txt").write_text(subprocess.check_output([python, "-m", "pip", "freeze"], text=True))
(bundle/"gpu.txt").write_text(subprocess.check_output(["nvidia-smi"], text=True))
(bundle/"machine.txt").write_text(subprocess.check_output(["lscpu"], text=True) + "\n" + Path("/proc/meminfo").read_text() + "\n" + Path("/etc/os-release").read_text())

In [ ]:
run("-m", "models.evaluate", "--device", "cuda", "--weights", str(bundle/"best.pth"), "--out", str(bundle/"evidence"))
import hashlib
with (bundle/"best.pth").open("rb") as f:
    (bundle/"sha256.txt").write_text(hashlib.file_digest(f, "sha256").hexdigest()+"  best.pth\n")
archive = shutil.make_archive("/content/envelope-training-evidence", "zip", bundle)
files.download(archive)